In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import lamindb as ln

In [ ]:
SAMPLE_NAMES = [
    'TIS04399-001',
    'TIS02434-001',
    'TIS02437-001',
    'TIS04398-001',
    'TIS02611-001',
    'TIS02603-001',
    'TIS02604-001',
    'TIS02613-001',
    'TIS02421-001',
    'TIS02430-001',
    'TIS02424-001',
    'TIS02439-001',
    'TIS02594-001',
]

samples = {}
for name in SAMPLE_NAMES:
    adata = ln.Artifact.get(key=f'{name}.h5ad').load()
    sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
    samples[name] = adata

In [ ]:
bin_edges  = [0, 5, 10, 15, 30, 60, np.inf]
bin_labels = ['0-4', '5-9', '10-14', '15-29', '30-59', '60+']

for adata in samples.values():
    adata.obs['count_bin'] = pd.cut(adata.obs['total_counts'], bins=bin_edges, labels=bin_labels, right=False).cat.remove_unused_categories()

pd.concat({
    name: adata.obs.groupby('count_bin', observed=True).agg(
        cells=('total_counts', 'size'),
        median_genes=('n_genes_by_counts', 'median'),
        median_area=('cell_area', 'median'),
    )
    for name, adata in samples.items()
})

In [ ]:
share_of_counts = {}
for name, adata in samples.items():
    expression  = pd.DataFrame(adata.X.toarray(), columns=adata.var_names).drop(columns='AZU1')
    gene_totals = expression.groupby(adata.obs['count_bin'].array, observed=True).sum()
    share_of_counts[name] = gene_totals.div(gene_totals.sum(axis=1), axis=0)

In [ ]:
ncols = 3
nrows = -(-len(share_of_counts) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 7 * nrows))

for ax, (name, share) in zip(axes.ravel(), share_of_counts.items()):
    top_genes = pd.DataFrame({label: share.loc[label].nlargest(10).index for label in share.index})
    shown = pd.unique(top_genes.to_numpy().ravel())
    image = ax.imshow(share[shown].T, aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(share)), share.index)
    ax.set_yticks(range(len(shown)), shown, fontsize=7)
    ax.set_xlabel('total_counts bin')
    ax.set_title(name)
    fig.colorbar(image, ax=ax, label='share of counts')

for ax in axes.ravel()[len(share_of_counts):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
THRESHOLD_A = {'min_counts': 15, 'min_genes': 5}
THRESHOLD_B = {'min_counts': 15, 'min_genes': 10}

CATEGORY_COLORS = {
    'kept by both': 'lightgrey',
    'removed only by A': 'tab:red',
    'removed only by B': 'tab:blue',
    'removed by both': 'black',
}


def passes(obs, threshold):
    return (obs['total_counts'] >= threshold['min_counts']) & (obs['n_genes_by_counts'] >= threshold['min_genes'])


for adata in samples.values():
    passes_a = passes(adata.obs, THRESHOLD_A)
    passes_b = passes(adata.obs, THRESHOLD_B)
    outcome = np.select(
        [passes_a & passes_b, ~passes_a & passes_b, passes_a & ~passes_b],
        ['kept by both', 'removed only by A', 'removed only by B'],
        default='removed by both',
    )
    # adds obs['filter_comparison'] (in memory only); .X is not changed. Plotting it adds uns['filter_comparison_colors']
    adata.obs['filter_comparison'] = pd.Categorical(outcome, categories=list(CATEGORY_COLORS))

ncols = 3
nrows = -(-len(samples) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))

for ax, (name, adata) in zip(axes.ravel(), samples.items()):
    outcome = adata.obs['filter_comparison']
    sc.pl.spatial(
        adata, color='filter_comparison', palette=list(CATEGORY_COLORS.values()), spot_size=25,
        ax=ax, show=False, legend_loc=None,
        title=f"{name}\nA removes {outcome.isin(['removed only by A', 'removed by both']).mean():.1%}, "
              f"B removes {outcome.isin(['removed only by B', 'removed by both']).mean():.1%}",
    )

for ax in axes.ravel()[len(samples):]:
    ax.set_visible(False)

handles = [plt.Line2D([], [], marker='o', linestyle='', color=color, label=label) for label, color in CATEGORY_COLORS.items()]
fig.legend(handles=handles, loc='upper right', title=f'A: {THRESHOLD_A}\nB: {THRESHOLD_B}')
fig.tight_layout()
plt.show()

pd.DataFrame({name: adata.obs['filter_comparison'].value_counts() for name, adata in samples.items()}).T

In [ ]:
from matplotlib.colors import LogNorm

ncols = 3
nrows = -(-len(samples) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))

for ax, (name, adata) in zip(axes.ravel(), samples.items()):
    ax.hist2d(
        adata.obs['total_counts'], adata.obs['n_genes_by_counts'],
        bins=[np.arange(0, 101), np.arange(0, 51)], norm=LogNorm(), cmap='Greys',
    )
    for threshold, color in [(THRESHOLD_A, 'tab:red'), (THRESHOLD_B, 'tab:blue')]:
        ax.axvline(threshold['min_counts'], color=color)
        ax.axhline(threshold['min_genes'], color=color)
    ax.set_xlabel('total_counts')
    ax.set_ylabel('n_genes_by_counts')
    ax.set_title(name)

for ax in axes.ravel()[len(samples):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
thresholds = {'A': THRESHOLD_A, 'B': THRESHOLD_B}

removed_gene_share = {}
for name, adata in samples.items():
    shares = {}
    for label, threshold in thresholds.items():
        removed = adata[~passes(adata.obs, threshold).to_numpy()]
        gene_totals = pd.Series(np.asarray(removed.X.sum(axis=0)).ravel(), index=adata.var_names).drop('AZU1')
        shares[f'removed by {label}'] = gene_totals / gene_totals.sum()
    removed_gene_share[name] = pd.DataFrame(shares)

ncols = 3
nrows = -(-len(removed_gene_share) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 7 * nrows))

for ax, (name, share) in zip(axes.ravel(), removed_gene_share.items()):
    shown = pd.unique(np.concatenate([share[column].nlargest(10).index for column in share]))
    image = ax.imshow(share.loc[shown], aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(share.columns)), share.columns)
    ax.set_yticks(range(len(shown)), shown, fontsize=7)
    ax.set_title(f'{name}\nA: {THRESHOLD_A}\nB: {THRESHOLD_B}', fontsize=8)
    fig.colorbar(image, ax=ax, label='share of counts')

for ax in axes.ravel()[len(removed_gene_share):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()